# La entrega frente a los 5 minutos públicos, en detalle

**Lo ejecuta una persona, en local.** Lee una entrega del partido (`evaluation/results/*.csv`), así que:

- **no se sube con outputs:** antes de cada commit,
  `python .claude/skills/subir/outputs_notebooks.py --limpiar notebooks/ventana_publica_detalle.ipynb`;
- **a Claude solo se le pegan agregados** (recuentos, porcentajes, métricas), nunca las tablas frame a frame ni los
  gráficos;
- **no sirve para elegir entre versiones ni para ajustar nada** (regla del repo): son 17 posesiones y el scorer dice que
  por debajo de 0,42 todo es ruido. Sirve para entender qué tipo de errores hacemos y para mirarlos en el vídeo.

Las métricas salen de `evaluation/possession_metrics.py`, el mismo código que el scorer oficial.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RAIZ = os.path.abspath(os.path.join(os.getcwd(), "..")) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, os.path.join(RAIZ, "evaluation"))
import possession_metrics as pm

FPS = 25
# Qué entregas mirar: la primera es la principal; las demás, referencias (las que no existan se saltan).
ENTREGAS = ["equipo_v10", "equipo_v9", "baseline_smooth50"]
CARPETA = os.path.join(RAIZ, "evaluation", "results")
GT = os.path.join(RAIZ, "evaluation", "gt", "BAR-ATH_possession_gt_public.csv")
TOLERANCIA = 25  # frames: un cambio nuestro empareja con uno real a ±1 s, como el transF1 oficial
COLORES = {"H": "#1f5fbf", "A": "#c8102e", "D": "#9a9a9a", "X": "#ffffff"}

In [ ]:
gt = pm.label_codes(pm.read_label_csv(GT))
runs = pm.valid_runs(gt)
preds = {}
for nombre in ENTREGAS:
    ruta = os.path.join(CARPETA, nombre + ".csv")
    if os.path.isfile(ruta):
        preds[nombre] = pm.label_codes(pm.read_label_csv(ruta))
    else:
        print("no está {}: se salta".format(nombre))
PRINCIPAL = next(iter(preds))
LETRA = dict(enumerate(pm.LABELS))
D = pm.LABELS.index("D")


def reloj(frame):
    s = frame / FPS
    return "{:02d}:{:05.2f}".format(int(s // 60), s % 60)


print("tramos evaluados en la ventana:", [(a, b, reloj(a), reloj(b)) for a, b in runs])
print("frames evaluados:", int(sum(b - a for a, b in runs)), "· entregas:", list(preds))

## 1. Las métricas oficiales

In [ ]:
filas = {}
for nombre, pred in preds.items():
    r = pm.evaluate(gt, pred, bootstrap_n=0)
    filas[nombre] = {k: v for k, v in pd.json_normalize(r, sep=".").iloc[0].items()
                     if isinstance(v, (int, float, np.integer, np.floating))}
resumen = pd.DataFrame(filas)
claves = [c for c in resumen.index if any(k in c for k in ("spell", "macro", "trans", "poss", "lat", "frag"))]
resumen.loc[claves]

## 2. Matriz de confusión (filas = la verdad, columnas = lo nuestro)

In [ ]:
def matriz(pred):
    cm = pm.confusion(gt, pred)[:3, :]  # filas H, A, D de la GT (los X de la GT no se puntúan)
    return pd.DataFrame(cm, index=["GT " + l for l in pm.LABELS[:3]], columns=["nuestro " + l for l in pm.LABELS])


for nombre, pred in preds.items():
    m = matriz(pred)
    print("\n" + nombre)
    display(m)
    display((m.div(m.sum(axis=1), axis=0) * 100).round(1).add_suffix(" %"))

fig, ejes = plt.subplots(1, len(preds), figsize=(4.2 * len(preds), 3.6))
for eje, (nombre, pred) in zip(np.atleast_1d(ejes), preds.items()):
    m = matriz(pred).iloc[:, :3].values
    fila = m / np.maximum(m.sum(axis=1, keepdims=True), 1)
    eje.imshow(fila, cmap="Blues", vmin=0, vmax=1)
    for i in range(3):
        for j in range(3):
            eje.text(j, i, "{}\n{:.0%}".format(m[i, j], fila[i, j]), ha="center", va="center", fontsize=9,
                     color="white" if fila[i, j] > 0.6 else "black")
    eje.set_xticks(range(3), pm.LABELS[:3])
    eje.set_yticks(range(3), pm.LABELS[:3])
    eje.set_xlabel("nuestro")
    eje.set_ylabel("verdad")
    eje.set_title(nombre)
plt.tight_layout()
plt.show()

## 3. La línea de tiempo: la verdad y cada entrega

In [ ]:
def tiras(codigos_por_nombre, a, b, titulo):
    fig, eje = plt.subplots(figsize=(16, 0.6 + 0.55 * len(codigos_por_nombre)))
    for fila, (nombre, codigos) in enumerate(codigos_por_nombre.items()):
        for ini, fin, k in pm.segments(codigos, [(a, b)]):
            eje.broken_barh([(ini / FPS, (fin - ini) / FPS)], (fila - 0.4, 0.8), color=COLORES[LETRA[k]])
    eje.set_yticks(range(len(codigos_por_nombre)), list(codigos_por_nombre))
    eje.invert_yaxis()
    eje.set_xlabel("segundo del vídeo")
    eje.set_title(titulo)
    plt.tight_layout()
    plt.show()


for a, b in runs:
    tiras(dict([("verdad", gt)] + list(preds.items())), a, b,
          "{} a {} (azul H, rojo A, gris D)".format(reloj(a), reloj(b)))

## 4. Los cambios de posesión: cuáles vemos, con cuánto retraso, y cuáles no

Un cambio real se empareja con uno nuestro **hacia el mismo estado** a ±1 s, como el transF1 oficial. Retraso positivo:
llegamos tarde.

In [ ]:
def cambios(pred):
    llenado = pm.fill_abstentions(pred, runs)
    reales, nuestros = pm.transitions(gt, runs), pm.transitions(llenado, runs)
    parejas, perdidos, sobrantes = pm.match_transitions(reales, nuestros, TOLERANCIA)
    filas = []
    for r, p in parejas:
        filas.append({"frame real": r[0], "vídeo": reloj(r[0]), "cambio": LETRA[r[1]] + "→" + LETRA[r[2]],
                      "nuestro frame": p[0], "retraso (frames)": p[0] - r[0], "estado": "visto"})
    for r in perdidos:
        filas.append({"frame real": r[0], "vídeo": reloj(r[0]), "cambio": LETRA[r[1]] + "→" + LETRA[r[2]],
                      "nuestro frame": np.nan, "retraso (frames)": np.nan, "estado": "no visto"})
    for p in sobrantes:
        filas.append({"frame real": np.nan, "vídeo": reloj(p[0]), "cambio": LETRA[p[1]] + "→" + LETRA[p[2]],
                      "nuestro frame": p[0], "retraso (frames)": np.nan, "estado": "nos sobra"})
    t = pd.DataFrame(filas)
    t["orden"] = t["frame real"].fillna(t["nuestro frame"])
    return t.sort_values("orden").drop(columns="orden").reset_index(drop=True)


tabla_cambios = cambios(preds[PRINCIPAL])
print(tabla_cambios["estado"].value_counts().to_dict())
vistos = tabla_cambios[tabla_cambios["estado"] == "visto"]
if len(vistos):
    print("retraso de los vistos (frames): mediana {:+.0f}, media {:+.1f}, máximo |{}|".format(
        vistos["retraso (frames)"].median(), vistos["retraso (frames)"].mean(),
        int(vistos["retraso (frames)"].abs().max())))
print("\npor tipo de cambio real:")
display(tabla_cambios[tabla_cambios["estado"] != "nos sobra"].groupby(["cambio", "estado"]).size()
        .unstack(fill_value=0))
tabla_cambios

In [ ]:
fig, eje = plt.subplots(figsize=(7, 3))
eje.hist(vistos["retraso (frames)"], bins=np.arange(-TOLERANCIA - 0.5, TOLERANCIA + 1.5, 2), color="#1f5fbf")
eje.axvline(0, color="black", lw=1)
eje.set_xlabel("retraso (frames; positivo = tarde)")
eje.set_ylabel("cambios")
eje.set_title("{}: retraso de los cambios que vemos".format(PRINCIPAL))
plt.tight_layout()
plt.show()

## 5. Las posesiones reales: cuáles encontramos y por qué perdemos las demás

Encontrada: emparejada con IoU ≥ 0,5 por el emparejamiento del scorer. La causa de las que no encontramos, en este orden
(la misma clasificación que `validation/scripts/diagnostico_errores.py`):
**la tapamos con D** (la mitad o más de sus frames los decimos `D`) · **otro equipo** (decimos el otro equipo en la
mayoría) · **la nuestra es más larga** (decimos el bueno, pero se funde con la de al lado) · **la nuestra es más corta o
va partida**.

In [ ]:
UMBRAL = 0.5


def posesiones(pred):
    llenado = pm.fill_abstentions(pred, runs)
    reales, nuestras = pm.possession_spells(gt, runs), pm.possession_spells(llenado, runs)
    solapes = pm.spell_overlaps(reales, nuestras)
    ok_real = pm.matched_gt_flags(solapes, len(reales), UMBRAL)
    ok_nuestra = np.zeros(len(nuestras), bool)
    usada_real = np.zeros(len(reales), bool)
    mejor_iou = np.zeros(len(reales))
    for iou, i, j in solapes:
        mejor_iou[i] = max(mejor_iou[i], iou)
        if iou >= UMBRAL and not usada_real[i] and not ok_nuestra[j]:
            usada_real[i] = ok_nuestra[j] = True

    def la_que_mas_pisa(a, b, k, otras):
        mejor, comun = None, 0
        for c, d, e in otras:
            n = min(b, d) - max(a, c)
            if e == k and n > comun:
                mejor, comun = (c, d, e), n
        return mejor

    filas = []
    for (a, b, k), ok, iou in zip(reales, ok_real, mejor_iou):
        dentro = llenado[a:b]
        if ok:
            causa = "encontrada"
        elif np.mean(dentro == D) >= 0.5:
            causa = "la tapamos con D"
        elif np.mean(dentro == k) < 0.5:
            causa = "otro equipo"
        else:
            m = la_que_mas_pisa(a, b, k, nuestras)
            causa = ("la nuestra es más larga" if m and (m[1] - m[0]) > (b - a)
                     else "la nuestra es más corta o va partida")
        filas.append({"equipo": LETRA[k], "inicio": reloj(a), "fin": reloj(b), "duración (s)": round((b - a) / FPS, 1),
                      "mejor IoU": round(float(iou), 2),
                      "frames con el equipo bueno %": round(100 * float(np.mean(dentro == k))),
                      "resultado": causa})
    sobran = [{"equipo": LETRA[k], "inicio": reloj(a), "fin": reloj(b), "duración (s)": round((b - a) / FPS, 1)}
              for (a, b, k), ok in zip(nuestras, ok_nuestra) if not ok]
    return pd.DataFrame(filas), pd.DataFrame(sobran)


tabla_pos, tabla_sobran = posesiones(preds[PRINCIPAL])
print("reales:", len(tabla_pos), "· encontradas:", int((tabla_pos["resultado"] == "encontrada").sum()),
      "· nuestras que sobran:", len(tabla_sobran))
display(tabla_pos["resultado"].value_counts().to_frame("posesiones"))
tabla_pos

In [ ]:
print("Nuestras posesiones que no emparejan con ninguna real:")
tabla_sobran

In [ ]:
tramos = [0, 1, 2, 5, 10, 1e9]
nombres = ["< 1 s", "1–2 s", "2–5 s", "5–10 s", "≥ 10 s"]
tabla_pos["duración"] = pd.cut(tabla_pos["duración (s)"], tramos, right=False, labels=nombres)
display(tabla_pos.groupby("duración", observed=False)["resultado"]
        .agg(reales="size", encontradas=lambda s: int((s == "encontrada").sum())))

fig, eje = plt.subplots(figsize=(7, 3))
dur_nuestras = [(b - a) / FPS for a, b, _ in pm.possession_spells(pm.fill_abstentions(preds[PRINCIPAL], runs), runs)]
eje.hist([tabla_pos["duración (s)"], dur_nuestras], bins=np.arange(0, 65, 5), label=["reales", PRINCIPAL],
         color=["#444444", "#1f5fbf"])
eje.set_xlabel("duración de la posesión (s)")
eje.set_ylabel("posesiones")
eje.legend()
plt.tight_layout()
plt.show()

## 6. Qué cambia entre entregas (en la ventana)

In [ ]:
evaluados = gt != pm.LABELS.index("X")
for otra in list(preds)[1:]:
    distinto = evaluados & (preds[PRINCIPAL] != preds[otra])
    print("\n{} frente a {}: {} de {} frames evaluados distintos".format(
        PRINCIPAL, otra, int(distinto.sum()), int(evaluados.sum())))
    if distinto.any():
        de_a = pd.Series([LETRA[x] + "→" + LETRA[y] for x, y in zip(preds[otra][distinto], preds[PRINCIPAL][distinto])])
        acierta = pd.Series(np.where(preds[PRINCIPAL][distinto] == gt[distinto], "acierta " + PRINCIPAL,
                                     np.where(preds[otra][distinto] == gt[distinto], "acierta " + otra, "ninguna")))
        display(pd.crosstab(de_a.rename("de {} a {}".format(otra, PRINCIPAL)), acierta))